In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Diabetes-Prediction"
)

DATA_PATH = (
    PROJECT_DIR / "data" /
    "project_data.xlsx"
)

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (20000, 15)


,Gender,Age,Physical Activity,Smoking Status,Alcohol Intake,Glucose,Blood Pressure,Skin Thickness,Insulin,BMI,Cholesterol,Diabetes Pedigree Function,Family History,Hypertension,Outcome
0,Male,63,Moderate,Former,NaN,103,40,7,41,28.196216,167,1.393,Yes,Yes,Diabetic
1,Female,51,Moderate,Never,Regular,104,59,21,51,30.438114,209,0.815,No,No,Diabetic
2,Female,74,Moderate,Current,NaN,128,70,18,199,38.700428,170,0.244,No,No,Diabetic
3,Male,78,Low,Never,Occasional,148,72,15,110,44.861600,170,0.336,No,No,Diabetic
4,Male,32,High,Never,NaN,102,55,18,49,21.591572,211,1.146,Yes,No,Non-diabetic


validating columns and target label
TARGET identifies the value the model must predict.
expected_columns defines the features we expect in this version of the dataset.
The set differences help identify missing or unexpected columns.
value_counts(dropna=False) displays the available labels, including missing targets.

In [4]:
TARGET = "Outcome"

expected_columns = [
    "Gender",
    "Age",
    "Physical Activity",
    "Smoking Status",
    "Alcohol Intake",
    "Glucose",
    "Blood Pressure",
    "Skin Thickness",
    "Insulin",
    "BMI",
    "Cholesterol",
    "Diabetes Pedigree Function",
    "Family History",
    "Hypertension",
    "Outcome"
]

missing_columns = set(expected_columns) - set(df.columns)
unexpected_columns = set(df.columns) - set(expected_columns)

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

if unexpected_columns:
    print("Review unexpected columns:", unexpected_columns)

print("Outcome labels:")
print(df[TARGET].value_counts(dropna=False))

Outcome labels:
Outcome
Diabetic        16726
Non-diabetic     3274
Name: count, dtype: int64


cleaning basic data quality problems ....like duplicates row and missing target value....


In [5]:
print("the duplicates row :",df.duplicated().sum())
print("missing target values :",df[TARGET].isna().sum())
df=df.drop_duplicates().copy()
df=df.dropna(subset=[TARGET]).copy()
# removes accidental leading or trailing spaces from target label
df[TARGET]=df[TARGET].astype(str).str.strip()
print("Shape after basic cleaning:",df.shape)
print(df[TARGET].value_counts)

the duplicates row : 0
missing target values : 0
Shape after basic cleaning: (20000, 15)
<bound method IndexOpsMixin.value_counts of 0            Diabetic
1            Diabetic
2            Diabetic
3            Diabetic
4        Non-diabetic
             ...     
19995        Diabetic
19996        Diabetic
19997        Diabetic
19998        Diabetic
19999        Diabetic
Name: Outcome, Length: 20000, dtype: object>


validating label

In [6]:
valid_labels = {"Diabetic", "Non-diabetic"}

actual_labels = set(df[TARGET].unique())

if not actual_labels.issubset(valid_labels):
    raise ValueError(
        f"Unexpected Outcome labels: {actual_labels}"
    )

print("Target labels validated.")
# This prevents the notebook from silently training on unexpected labels.
#If your file uses different label spellings, inspect and standardize them explicitly rather than guessing.


Target labels validated.


Input  feature or target ko alag alag kroo.....

In [7]:
numerical_cols = [
    "Age",
    "Glucose",
    "Blood Pressure",
    "Skin Thickness",
    "Insulin",
    "BMI",
    "Cholesterol",
    "Diabetes Pedigree Function"
]
binary_cols=[
    "Family History",
    "Hypertension"
]
categorical_cols = [
    "Gender",
    "Physical Activity",
    "Smoking Status",
    "Alcohol Intake",

]
TARGET="Outcome"

# Normalize binary column values
for col in binary_cols:
    df[col] = df[col].apply(
        lambda x: x.strip().lower()
        if isinstance(x, str) else x
    )

# Validate categories before mapping
for col in binary_cols:
    observed = set(df[col].dropna().unique())
    if not observed.issubset({"yes", "no"}):
        raise ValueError(
            f"Unexpected values in {col}: {observed}"
        )

# Convert Yes/No to 1/0
binary_mapping = {
    "yes": 1,
    "no": 0
}

for col in binary_cols:
    df[col] = df[col].map(binary_mapping)

# Verify the conversion
for col in binary_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))
X = df[numerical_cols + binary_cols + categorical_cols].copy()
y = df[TARGET].map({
    "Diabetic": 1,
    "Non-diabetic": 0
})

print("Input features:", X.shape)
print("Target:", y.shape)

print("\nNumerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))


Family History
Family History
0    12083
1     7917
Name: count, dtype: int64

Hypertension
Hypertension
0    13974
1     6026
Name: count, dtype: int64
Input features: (20000, 14)
Target: (20000,)

Numerical features: 8
Categorical features: 4


Spliting the data set in training and testing data

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining outcome distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting outcome distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (16000, 14)
Testing features: (4000, 14)

Training outcome distribution:
Outcome
1    0.836
0    0.164
Name: proportion, dtype: float64

Testing outcome distribution:
Outcome
1    0.836
0    0.164
Name: proportion, dtype: float64


Building preprocessing pipeline ...

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler

# making pipeline for numerical columns
numerical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])
categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="constant",fill_value="None")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])
preprocessor=ColumnTransformer([
    ("numerical",numerical_pipeline,numerical_cols),
     ("binary", binary_pipeline, binary_cols),
    ("categorical",categorical_pipeline,categorical_cols)
])
print("Preprocessing pipeline created....")

Preprocessing pipeline created....


Applying preprocessing on training data...

In [10]:
X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)
print("Original training shape:",X_train.shape)
print("Processed training shape:",X_train_processed.shape)
print("Original testing shape:",X_test.shape)
print("Processed testing shape:",X_test_processed.shape)
print(X_train)
print(X_train_processed)
print(X_test)
print(X_test_processed)

Original training shape: (16000, 14)
Processed training shape: (16000, 21)
Original testing shape: (4000, 14)
Processed testing shape: (4000, 21)
       Age  Glucose  Blood Pressure  Skin Thickness  Insulin        BMI  \
7953    57      124              69              10       51  29.731695   
2579    45      155              68               7       31  37.467824   
17493   71       98              77              18       79  40.242030   
13208   77      152              80              17       15  41.218966   
152     32       98              71              18       75  33.870652   
...    ...      ...             ...             ...      ...        ...   
7151    30       83              71              12       68  20.387780   
12808   39       86              53              14       55  36.425863   
2379    78      105              53              23      111  37.974915   
4706    64      130              68              14      199  38.710056   
14814   55       98          

In [11]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nFirst 20 feature names:")
print(feature_names[:23])

Number of processed features: 21

First 20 feature names:
['numerical__Age' 'numerical__Glucose' 'numerical__Blood Pressure'
 'numerical__Skin Thickness' 'numerical__Insulin' 'numerical__BMI'
 'numerical__Cholesterol' 'numerical__Diabetes Pedigree Function'
 'binary__Family History' 'binary__Hypertension'
 'categorical__Gender_Female' 'categorical__Gender_Male'
 'categorical__Physical Activity_High'
 'categorical__Physical Activity_Low'
 'categorical__Physical Activity_Moderate'
 'categorical__Smoking Status_Current'
 'categorical__Smoking Status_Former' 'categorical__Smoking Status_Never'
 'categorical__Alcohol Intake_None'
 'categorical__Alcohol Intake_Occasional'
 'categorical__Alcohol Intake_Regular']


In [12]:
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Diabetes-Prediction")
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

def to_dense_array(data):
    return data.toarray() if hasattr(data, "toarray") else data

X_train_array = to_dense_array(X_train_processed)
X_test_array = to_dense_array(X_test_processed)

joblib.dump(
    preprocessor,
    ARTIFACTS_DIR / "preprocessor.joblib"
)

pd.DataFrame(
    X_train_array, columns=feature_names
).to_csv(ARTIFACTS_DIR / "X_train_processed.csv", index=False)

pd.DataFrame(
    X_test_array, columns=feature_names
).to_csv(ARTIFACTS_DIR / "X_test_processed.csv", index=False)

y_train.to_frame(name=TARGET).to_csv(
    ARTIFACTS_DIR / "y_train.csv", index=False
)

y_test.to_frame(name=TARGET).to_csv(
    ARTIFACTS_DIR / "y_test.csv", index=False
)

print("Preprocessing completed and files saved.")

Preprocessing completed and files saved.
